# RTStream V2 Cookbook

Use VideoDB to connect a live RTSP feed, understand it continuously, turn the understanding into a searchable index, and react to events in real time.

The required path is:

**connect to VideoDB → connect the stream → create an understanding → read records → create an index → search → stop resources**

Alerts, pause/resume, and recording export are clearly marked as optional. Run the required sections from top to bottom. Live resources consume compute while they are running, so always run **Stop and clean up** before leaving the notebook.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/indexing-v2/rtstream/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1. Prerequisites**

You need a [VideoDB API key](https://console.videodb.io/) and an RTSP source. A public sample stream is included below.

**Install the RTStream V2 SDK**

Run this once per notebook session. If the notebook asks you to restart the kernel after installation, restart it and continue with the next section.

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/video-db/videodb-python.git@feat/add-indexing-v2" python-dotenv

**Connect to VideoDB**

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")
if not api_key:
    raise ValueError("A VideoDB API key is required.")

conn = connect(api_key=api_key)
collection = conn.get_collection()

print("Connected to VideoDB")
print("Collection ID:", collection.id)

Connected to VideoDB
Collection ID: c-743871c9-481c-4630-a55b-004f8d224735


**Choose an RTSP source**

RTSP (Real-Time Streaming Protocol) is commonly used by live cameras and streaming servers. VideoDB pulls media from the URL, so a custom source must be reachable from the internet—not `localhost` or a private network address.

Use either public sample below, or replace `RTSP_URL` with your own feed:

- Crib: `rtsp://samples.rts.videodb.io:8554/crib`
- Cricket: `rtsp://samples.rts.videodb.io:8554/cricket`
- Custom: `rtsp://your-public-host:port/path`

**Connect the RTStream**

`connect_rtstream()` connects the source and returns an RTStream with an `rts-` ID.

- `STORE_RECORDING` retains the live media so it can be exported after the stream stops. Set it to `False` when you do not need the recording.

In [133]:
import time
from datetime import datetime, timezone

RTSP_URL = "rtsp://samples.rts.videodb.io:8554/crib"
# RTSP_URL = "rtsp://samples.rts.videodb.io:8554/cricket"
# RTSP_URL = "rtsp://your-public-host:port/path"
STREAM_NAME = "rtstream-v2-cookbook"

STORE_RECORDING = True   # Retain media so the stopped stream can be exported.

media_types = ["video"]
run_started_at = time.time()

rtstream = collection.connect_rtstream(
    url=RTSP_URL,
    name=f"{STREAM_NAME}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}",
    media_types=media_types,
    store=STORE_RECORDING,
)

print("RTStream ID:", rtstream.id)
print("Status:", rtstream.status)
print("Source:", RTSP_URL)
print("Media types:", media_types)

RTStream ID: rts-019f8bc5-9811-78d1-9eec-fda576c5fad2
Status: connected
Source: rtsp://samples.rts.videodb.io:8554/crib
Media types: ['video']


### **2. Understand the live stream**

**Create a continuous understanding**

Understanding continuously turns live video into time-aligned descriptions. For each window, VideoDB samples frames, analyzes them with the VLM and your prompt, and stores the result so it can power records, indexing, search, and alerts.

The settings in this cell apply specifically to `rtstream.understand()`:

- `WINDOW` is the duration of each understanding segment. `"10s"` means the VLM analyzes one 10-second portion of the stream at a time. Shorter windows produce updates more frequently.
- `FRAME_COUNT` is the number of frames sampled evenly across each window and sent to the VLM. With a 10-second window and 5 frames, samples are spaced about 2.5 seconds apart. More frames provide more visual context but require more processing.
- `SCENE_PROMPT` tells the VLM what to describe in every window.

The VLM writes each response to the named output `scene`. The 10-second/5-frame defaults are a practical starting point.

`store=True` is important in this workflow: it makes understanding records durable and allows an index to consume the output.

In [134]:
WINDOW = "10s"          # Analyze the stream in 10-second segments.
FRAME_COUNT = 5          # Sample 5 frames evenly across each segment.
SCENE_PROMPT = "Describe the scene clearly. Mention whether a baby or crib is visible and what is happening."

understanding = rtstream.understand(
    segmentation={"type": "time", "window": WINDOW},
    analyzers=[
        {
            "type": "vlm",
            "name": "scene",
            "sampling": {"frame_count": FRAME_COUNT},
            "config": {"prompt": SCENE_PROMPT},
        }
    ],
    store=True,
)

print("Understanding ID:", understanding.id)
print("Status:", understanding.status)
print("Available outputs:", list(understanding.outputs))
print("Scene output descriptor:", understanding.outputs["scene"])

Understanding ID: und-cbec401e3b562e1b
Status: running
Available outputs: ['scene']
Scene output descriptor: {'asset_type': 'rtstream', 'extract_type': 'vlm', 'mode': 'continuous', 'output': 'scene', 'rtstream_id': 'rts-019f8bc5-9811-78d1-9eec-fda576c5fad2', 'type': 'understanding', 'understanding_id': 'und-cbec401e3b562e1b'}


**Retrieve an existing understanding**

In [135]:
same_understanding = rtstream.get_understanding(understanding.id)
all_understandings = rtstream.list_understanding()

print("Fetched ID:", same_understanding.id)
print("Understanding IDs on this stream:", [item.id for item in all_understandings])

Fetched ID: und-cbec401e3b562e1b
Understanding IDs on this stream: ['und-cbec401e3b562e1b']


**Read durable understanding records**

Records remain available after the RTStream stops. If `records` is empty, wait for another window and rerun the cell.

In [137]:
understanding_records = understanding.get_records(
    start=run_started_at,
    end=time.time(),
    output="scene",
)
understanding_records

{'next_page': False,
 'output': 'scene',
 'records': [{'data': {'scene_description': 'A baby/toddler is definitely visible, and a crib is also visible. The scene shows the child climbing over the top rail of the crib and then crawling out onto the floor. The room looks like a nursery or child’s bedroom, with toys and bedding around the crib.'},
   'end': 1784756427.423222,
   'metadata': {'frame_count': 5,
    'frame_timestamps': [1784756416.493225,
     1784756419.4232242,
     1784756422.4432235,
     1784756425.4532225,
     1784756427.423222],
    'meta': {'model': 'basic',
     'timestamp': '2026-07-22T21:40:30.691378',
     'usage': {'completion_tokens': 60,
      'cost_usd': 0,
      'prompt_tokens': 5614,
      'total_tokens': 5674}}},
   'scene_id': 'seg-rts-019f8bc5-9811-78d1-9eec-fda576c5fad2-und-cbec401e3b562e1b-1784756416493-1784756427423',
   'start': 1784756416.493225},
  {'data': {'scene_description': 'A toddler is in a bedroom, trying to climb out of a wooden crib/play

### **3. Index the understanding**

An index turns a named understanding output into records that can be searched.

**Create an index**

`use_for=["semantic"]` enables semantic search on the `scene` output.

In [138]:
index = rtstream.index(
    source=understanding.outputs["scene"],
    name="rtstream-v2-scenes",
    use_for=["semantic"],
)
index

RTStreamIndex(id=idx-5df42ae3ac1501a1, rtstream_id=rts-019f8bc5-9811-78d1-9eec-fda576c5fad2, status=running, use_for=['semantic'], source_understanding_id=und-cbec401e3b562e1b)

**Retrieve existing indexes**

In [139]:
same_index = rtstream.get_index(index.id)
all_indexes = rtstream.list_indexes()

same_index, all_indexes

(RTStreamIndex(id=idx-5df42ae3ac1501a1, rtstream_id=rts-019f8bc5-9811-78d1-9eec-fda576c5fad2, status=running, use_for=['semantic'], source_understanding_id=und-cbec401e3b562e1b),
 [RTStreamIndex(id=idx-5df42ae3ac1501a1, rtstream_id=rts-019f8bc5-9811-78d1-9eec-fda576c5fad2, status=running, use_for=['semantic'], source_understanding_id=und-cbec401e3b562e1b)])

**Read index records**

If `records` is empty, wait for another understanding window and rerun the cell.

In [140]:
index_records = index.get_records(
    start=run_started_at,
    end=time.time(),
)
index_records

{'next_page': False,
 'records': [{'description': 'A baby/toddler is definitely visible, and a crib is also visible. The scene shows the child climbing over the top rail of the crib and then crawling out onto the floor. The room looks like a nursery or child’s bedroom, with toys and bedding around the crib.',
   'end': 1784756427.423222,
   'start': 1784756416.493225},
  {'description': 'A toddler is in a bedroom, trying to climb out of a wooden crib/playpen. In the first scene, a wooden crib is clearly visible, and the child is hoisting himself over the top rail. In the second scene, the baby/young child is inside a mesh-sided playpen, again climbing up and leaning over the edge as if preparing to get out. No adult is visible, and the action is the child’s repeated escape attempt.',
   'end': 1784756439.4832187,
   'start': 1784756428.4632218},
  {'description': 'The scene shows a young toddler climbing around and over the side of a playpen/crib-like enclosure in a bedroom. No baby cr

### **4. Search the live index**

Search uses natural language to find matching time ranges in the index.

**Search for a scene**

In [144]:
SEARCH_QUERY = "a baby or crib is visible"

search_result = rtstream.search(
    query=SEARCH_QUERY,
    index_id=index.id,
    result_threshold=5,
)
shots = search_result.get_shots()
shots

[RTStreamShot(rtstream_id=rts-019f8bc5-9811-78d1-9eec-fda576c5fad2, rtstream_name=rtstream-v2-cookbook-20260722-214009, start=1784756416.493225, end=1784756427.423222, text=A baby/toddler is definitely visible, and a crib is also visible. The scene shows the child climbing over the top rail of the crib and then crawling out onto the floor. The room looks like a nursery or child’s bedroom, with toys and bedding around the crib., search_score=0.70334476, scene_index_id=5df42ae3ac1501a1),
 RTStreamShot(rtstream_id=rts-019f8bc5-9811-78d1-9eec-fda576c5fad2, rtstream_name=rtstream-v2-cookbook-20260722-214009, start=1784756441.443218, end=1784756452.453215, text=The scene shows a young toddler climbing around and over the side of a playpen/crib-like enclosure in a bedroom. No baby crib with an infant is visible in the first image, but later a crib is clearly visible in the final images, with a small child standing inside it. The child appears to be trying to climb out or move over the crib ra

**Play a result**

Run this after the search returns at least one shot.

In [145]:
first_shot = shots[0]
first_shot.play()

'https://console.videodb.io/player?url=https://rt.stream.videodb.io/manifests/rts-019f8bc5-9811-78d1-9eec-fda576c5fad2/1784756416000000-1784756427000000.m3u8'

### **5. Receive webhook alerts (optional)**

VideoDB sends a callback when a new indexed scene matches the event prompt.

**Start a webhook receiver**

This creates a temporary [Cloudflare Quick Tunnel](https://developers.cloudflare.com/tunnel/setup/#quick-tunnels-development). No Cloudflare account or token is required. Keep the notebook running.

In [147]:
!pip install -q flask pycloudflared


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [149]:
import threading
from flask import Flask, request
from pycloudflared import try_cloudflare
from werkzeug.serving import make_server

received_alerts = []
app = Flask("rtstream-alerts")


@app.post("/alerts")
def receive_alert():
    payload = request.get_json(silent=True) or {}
    received_alerts.append(payload)
    print("Alert received:", payload)
    return {"received": True}

webhook_server = make_server("0.0.0.0", 5002, app)
threading.Thread(target=webhook_server.serve_forever, daemon=True).start()
webhook_tunnel = try_cloudflare(port=5002, verbose=False)
CALLBACK_URL = f"{webhook_tunnel.tunnel}/alerts"
print("Callback URL:", CALLBACK_URL)

Download cloudflared...:   0%|          | 0/18959119 [00:00<?, ?it/s]

Callback URL: https://stability-money-institutions-orleans.trycloudflare.com/alerts


**Create an event and alert**

In [150]:
EVENT_PROMPT = "A baby crib is visible in the frame"
EVENT_LABEL = "crib-visible"

event_id = conn.create_event(
    event_prompt=EVENT_PROMPT,
    label=EVENT_LABEL,
)
alert_id = index.create_alert(
    event_id=event_id,
    callback_url=CALLBACK_URL,
)

{"event_id": event_id, "alert_id": alert_id}

{'event_id': '3c54cf7b1ce1a19a', 'alert_id': '89a9e9157e45f8f1'}

**View received callbacks**

In [156]:
received_alerts

[]

### **6. Optional — pause and resume processing**

An understanding and its index have independent lifecycles. Stopping either job pauses new processing but keeps existing records. This demonstration is opt-in because pausing the live pipeline creates a gap in new results.

In [158]:
RUN_PAUSE_RESUME_DEMO = True

if not RUN_PAUSE_RESUME_DEMO:
    print("Set RUN_PAUSE_RESUME_DEMO=True to run this lifecycle demonstration.")
else:
    index.stop()
    understanding.stop()
    print("Paused index and understanding")

    understanding.start()
    index.start()
    print("Resumed understanding and index")
    print("Understanding status:", rtstream.get_understanding(understanding.id).status)
    print("Index status:", rtstream.get_index(index.id).status)

Paused index and understanding
Resumed understanding and index
Understanding status: running
Index status: running


### **🚨 7. Stop and clean up — always run this**

Run this cell even if an earlier step failed. It attempts each cleanup action independently in this order:

1. disable the alert;
2. stop the index and understanding;
3. stop the RTStream;
4. close the optional webhook receiver.

Stopping compute does not delete stored understanding or index records.

In [161]:
cleanup_results = {}

if globals().get("alert_id"):
    try:
        index.disable_alert(alert_id)
        cleanup_results["alert"] = "disabled"
    except Exception as exc:
        cleanup_results["alert"] = f"disable failed: {exc}"

for label, resource in (
    ("index", globals().get("index")),
    ("understanding", globals().get("understanding")),
    ("rtstream", globals().get("rtstream")),
):
    if resource is None:
        continue
    try:
        resource.stop()
        cleanup_results[label] = "stopped"
    except Exception as exc:
        cleanup_results[label] = f"stop failed: {exc}"

if globals().get("webhook_tunnel"):
    try:
        try_cloudflare.terminate(5000)
        cleanup_results["tunnel"] = "stopped"
    except Exception as exc:
        cleanup_results["tunnel"] = f"stop failed: {exc}"

if globals().get("webhook_server"):
    try:
        webhook_server.shutdown()
        cleanup_results["webhook"] = "stopped"
    except Exception as exc:
        cleanup_results["webhook"] = f"stop failed: {exc}"

print(cleanup_results)

{'alert': 'disabled', 'index': 'stopped', 'understanding': 'stopped', 'rtstream': 'stop failed: Invalid request: RTStream status is already stopped ', 'tunnel': 'stop failed: port 5000 is not running.', 'webhook': 'stopped'}


### **8. Verify stored data after stopping**

Stored understanding and index records remain available after live compute stops.

**Read stored resources and records**

In [160]:
def summarize(resource):
    return {
        "id": getattr(resource, "id", None),
        "name": getattr(resource, "name", None),
        "status": getattr(resource, "status", None),
    }


def safe_read(label, fetch):
    try:
        value = fetch()
        print(f"{label}: OK")
        return value
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        print(f"{label}: {error}")
        return {"error": error}


post_stop_end = time.time()
post_stop_reads = {
    "stream": safe_read("stream", lambda: summarize(collection.get_rtstream(rtstream.id))),
    "understandings": safe_read(
        "understandings",
        lambda: [summarize(item) for item in rtstream.list_understanding()],
    ),
    "understanding": safe_read(
        "understanding",
        lambda: summarize(rtstream.get_understanding(understanding.id)),
    ),
    "understanding_records": safe_read(
        "understanding records",
        lambda: understanding.get_records(
            start=run_started_at,
            end=post_stop_end,
            output="scene",
            page_size=100,
        ),
    ),
    "indexes": safe_read("indexes", lambda: [summarize(item) for item in rtstream.list_indexes()]),
    "index": safe_read("index", lambda: summarize(rtstream.get_index(index.id))),
    "index_records": safe_read(
        "index records",
        lambda: index.get_records(start=run_started_at, end=post_stop_end, page_size=100),
    ),
    "alerts": safe_read("alerts", lambda: index.list_alerts()),
}

print("Post-stop reads complete. Use post_stop_reads to inspect full payloads.")

stream: OK
understandings: OK
understanding: OK
understanding records: OK
indexes: OK
index: OK
index records: OK
alerts: OK
Post-stop reads complete. Use post_stop_reads to inspect full payloads.


### **9. Export the retained recording (optional)**

This requires `STORE_RECORDING=True` and a stopped RTStream. The cell retries while the recording finishes processing.

**Export the recording**

In [ ]:
if not STORE_RECORDING:
    print("Skipped because this RTStream was created with STORE_RECORDING=False.")
else:
    export_result = None
    last_export_error = None

    for attempt in range(1, 7):
        try:
            export_result = rtstream.export(name="RTStream V2 Cookbook Recording")
            break
        except Exception as exc:
            last_export_error = exc
            print(f"Export attempt {attempt}/6 is not ready: {exc}")
            time.sleep(5)

    if export_result is None:
        print("The recording did not finalize during the retry window:", last_export_error)
    else:
        print("Video ID:", export_result.video_id)
        print("Duration:", export_result.duration)
        print("Media URL:", export_result.stream_url)
        print("Player URL:", export_result.player_url)

## Quick reference

| Goal | Public SDK call |
|---|---|
| Connect a live source | `collection.connect_rtstream(...)` |
| Start understanding | `rtstream.understand(...)` |
| Retrieve understandings | `rtstream.get_understanding(id)` / `rtstream.list_understanding()` |
| Read understanding output | `understanding.get_records(start, end, output="scene")` |
| Create a V2 index | `rtstream.index(source=understanding.outputs["scene"])` |
| Retrieve V2 indexes | `rtstream.get_index(id)` / `rtstream.list_indexes()` |
| Read indexed records | `index.get_records(start, end)` |
| Search | `rtstream.search(query, index_id=index.id)` |
| Create and control an alert | `index.create_alert(...)`, `disable_alert(id)`, `enable_alert(id)` |
| Generate a playable search result | `shot.generate_stream()` |
| Export a retained recording | `rtstream.export()` after `rtstream.stop()` |

## What you built

You connected a live source, created continuous understanding, stored its output in a searchable index, and read durable records.

You also searched live scenes and saw how optional alerts, clip generation, recording export, and lifecycle controls fit into the workflow.